In [2]:
from MicroTensorGrad.Engine import MicroTensor
from MicroTensorGrad.MLP import MLP
from MicroTensorGrad.utils import create_Batches
from MicroTensorGrad.Optimizers import Adam

In [3]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from copy import deepcopy

In [4]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')

In [5]:
mnist["data"].shape

(70000, 784)

In [6]:
mnist["target"].shape

(70000,)

In [7]:
X = mnist["data"]
Y = mnist["target"]

In [8]:
x, x_test, y, y_test = train_test_split(X, Y, test_size=0.2, stratify=Y)

In [9]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, stratify=y)

In [10]:
x_train.shape

(44800, 784)

In [11]:
y_train.shape

(44800,)

In [12]:
np.unique(y_train)

array(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'], dtype=object)

In [13]:
def vectorize_y(y):
    out = []
    for value in y:
        vec = np.zeros((10, ))
        vec[int(value)] = 1
        out.append(vec.copy())
    return np.array(out)

In [14]:
y_train = vectorize_y(y_train)
y_val = vectorize_y(y_val)
y_test = vectorize_y(y_test)

In [15]:
x_train = x_train[..., None]
x_val = x_val[..., None]
x_test = x_test[..., None]

y_train = y_train[..., None]
y_val = y_val[..., None]
y_test = y_test[..., None]

In [16]:
optimizer = Adam(lr=0.001)

In [17]:
mnist_model = MLP(
    components=[
        ["Linear", 784, 512],
        ["Tanh"],
        ["Linear", 512, 10],
    ],
    LossFunction="CrossEntropy",
    optimizer=optimizer
)

[['Linear', 784, 512], ['Tanh'], ['Linear', 512, 10]]
in : 784, out : 512
in : 512, out : 10


In [18]:
BatchSize = 1024

In [19]:
x_train_batches, y_train_batches = create_Batches(x_train, y_train, bacthSize=BatchSize)

In [20]:
epochs = 12

In [21]:
x_val_MicroTensor = MicroTensor(x_val)
x_train_MicroTensor = MicroTensor(x_train)

In [22]:
def accuracy(y_true, y_pred):
    y_true1 = y_true.reshape(y_true.shape[0], -1)
    y_pred1 = y_pred.reshape(y_true.shape[0], -1)
    
    return (y_true1.argmax(axis=1) == y_pred1.argmax(axis=1)).sum()/len(y_true1)

In [23]:
best_val_accuracy = -np.inf
best_weights = None
for i in range(epochs):
    mnist_model.training = True
    for x_batch, y_batch in zip(x_train_batches, y_train_batches):
        x_microTensor = MicroTensor(x_batch)
        topo = mnist_model(x_microTensor, y_batch)
        topo.backward()
        mnist_model.update()
        
        mnist_model.zero_grad()
    
    val_loss = mnist_model(x_val_MicroTensor, y_val).topo[-1].array[0]
    train_loss = mnist_model(x_train_MicroTensor, y_train).topo[-1].array[0]
    
    mnist_model.training = False
    val_accuracy = accuracy(y_val, mnist_model(x_val_MicroTensor))
    if val_accuracy > best_val_accuracy:
        best_weights = deepcopy(mnist_model.steps)
        best_val_accuracy = val_accuracy
    print(f"epoch {i}, train loss : {train_loss}, train accuracy : {accuracy(y_train, mnist_model(x_train_MicroTensor))} | val loss : {val_loss}, val accuracy : {val_accuracy}")

epoch 0, train loss : 0.39976576262916313, train accuracy : 0.8776339285714285 | val loss : 0.4126632372796026, val accuracy : 0.8707142857142857
epoch 1, train loss : 0.2970848348333704, train accuracy : 0.9077008928571428 | val loss : 0.30980931080754737, val accuracy : 0.9032142857142857
epoch 2, train loss : 0.25579690898449203, train accuracy : 0.9233258928571428 | val loss : 0.2720713404121321, val accuracy : 0.9158928571428572
epoch 3, train loss : 0.22663635228353257, train accuracy : 0.9313616071428571 | val loss : 0.2455942972210496, val accuracy : 0.924375
epoch 4, train loss : 0.2222099570507932, train accuracy : 0.9323660714285714 | val loss : 0.24206062894882546, val accuracy : 0.9271428571428572
epoch 5, train loss : 0.2064206790274906, train accuracy : 0.9378794642857143 | val loss : 0.2274857561866836, val accuracy : 0.93
epoch 6, train loss : 0.20131896786885917, train accuracy : 0.9387723214285715 | val loss : 0.22039809717824435, val accuracy : 0.9333035714285715
ep

In [24]:
mnist_model.steps = best_weights

In [25]:
mnist_model.training = False

In [26]:
accuracy(y_test, mnist_model(MicroTensor(x_test)))

np.float64(0.939)